# Global Shipping Chokepoints — 02: Find and Label the Real Chokepoints

Notebook 01 showed something important: the *globally* busiest 0.05-degree cells on
Earth are not Suez or Malacca, they're the Danish Straits (heavy short-haul ferry and
regional traffic inflates raw position counts there). Raw traffic density measures how
often a ship passed, not how much trade was riding on it — so instead of trusting a
naive "top N density cells" ranking, this notebook looks up the world's known strategic
trade chokepoints by their real geographic coordinates, and pulls each one's *actual*
measured traffic density from the same real data. That's the honest way to build the
comparison this project is actually about.

In [0]:
traffic = spark.table("workspace.global_shipping.traffic_density_grid")
ports = spark.table("workspace.global_shipping.world_port_index")

print(f"Traffic grid: {traffic.count():,} cells")
print(f"World Port Index: {ports.count():,} ports")
ports.printSchema()

Traffic grid: 3,040,763 cells
World Port Index: 3,669 ports
root
 |-- FID: integer (nullable = true)
 |-- INDEX_NO: integer (nullable = true)
 |-- REGION_NO: integer (nullable = true)
 |-- PORT_NAME: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- LAT_DEG: integer (nullable = true)
 |-- LAT_MIN: integer (nullable = true)
 |-- LAT_HEMI: string (nullable = true)
 |-- LONG_DEG: integer (nullable = true)
 |-- LONG_MIN: integer (nullable = true)
 |-- LONG_HEMI: string (nullable = true)
 |-- PUB: string (nullable = true)
 |-- CHART: string (nullable = true)
 |-- HARBORSIZE: string (nullable = true)
 |-- HARBORTYPE: string (nullable = true)
 |-- SHELTER: string (nullable = true)
 |-- ENTRY_TIDE: string (nullable = true)
 |-- ENTRYSWELL: string (nullable = true)
 |-- ENTRY_ICE: string (nullable = true)
 |-- ENTRYOTHER: string (nullable = true)
 |-- OVERHD_LIM: string (nullable = true)
 |-- CHA

## Known strategic chokepoints

Real, publicly documented coordinates for the world's major shipping chokepoints —
the narrow passages a huge share of global seaborne trade has no practical alternative
to using. Bounding boxes are drawn generously around each strait/canal's real extent so
the aggregation below catches the actual shipping lane, not just its exact center point.
The Danish Straits are included because notebook 01 showed real, heavy traffic there —
it's the entrance to the whole Baltic Sea, so it belongs in this comparison too.

In [0]:
# (name, lat_min, lat_max, lon_min, lon_max)
CHOKEPOINTS = [
    ("Strait of Malacca",   1.0,  6.5,  98.0, 104.5),
    ("Suez Canal",         29.8, 31.5,  32.1,  32.6),
    ("Panama Canal",        8.8,  9.4, -80.1, -79.5),
    ("Strait of Hormuz",   25.8, 27.2,  55.5,  57.0),
    ("Bab-el-Mandeb",      11.8, 13.5,  42.5,  44.0),
    ("Danish Straits",     54.5, 58.0,   9.5,  13.5),
    ("Strait of Dover",    50.7, 51.3,   1.0,   2.2),
    ("Strait of Gibraltar",35.7, 36.2,  -5.9,  -5.2),
    ("Bosphorus Strait",   40.9, 41.3,  28.8,  29.3),
]

In [0]:
from pyspark.sql import functions as F

rows = []
for name, lat_min, lat_max, lon_min, lon_max in CHOKEPOINTS:
    agg = (
        traffic.filter(
            (F.col("lat") >= lat_min) & (F.col("lat") <= lat_max)
            & (F.col("lon") >= lon_min) & (F.col("lon") <= lon_max)
        )
        .agg(
            F.sum("traffic_density").alias("total_traffic"),
            F.count("*").alias("n_cells"),
        )
        .collect()[0]
    )
    rows.append({
        "chokepoint": name,
        "total_traffic": agg["total_traffic"] or 0,
        "n_cells_with_traffic": agg["n_cells"],
        "lat_min": lat_min, "lat_max": lat_max, "lon_min": lon_min, "lon_max": lon_max,
    })

import pandas as pd

chokepoints_df = pd.DataFrame(rows).sort_values("total_traffic", ascending=False).reset_index(drop=True)
chokepoints_df["share_of_ranked_total"] = chokepoints_df["total_traffic"] / chokepoints_df["total_traffic"].sum()
display(chokepoints_df)

chokepoint,total_traffic,n_cells_with_traffic,lat_min,lat_max,lon_min,lon_max,share_of_ranked_total
Danish Straits,6408074441700,2841,54.5,58.0,9.5,13.5,0.6947721172682166
Strait of Malacca,1286868516300,5553,1.0,6.5,98.0,104.5,0.13952402891848564
Strait of Hormuz,594608248500,547,25.8,27.2,55.5,57.0,0.06446823230815885
Strait of Dover,471353840400,200,50.7,51.3,1.0,2.2,0.05110482230764081
Strait of Gibraltar,155948076200,93,35.7,36.2,-5.9,-5.2,0.016908101812973853
Bab-el-Mandeb,151830609400,321,11.8,13.5,42.5,44.0,0.01646168048119253
Suez Canal,75987978700,95,29.8,31.5,32.1,32.6,0.00823871965418762
Bosphorus Strait,71200697400,43,40.9,41.3,28.8,29.3,0.007719676126366622
Panama Canal,7402795500,33,8.8,9.4,-80.1,-79.5,8.026211227774331E-4


## Label each chokepoint with its nearest real port

Pulls the World Port Index port closest to each chokepoint's center, so the final
output reads with a real place name (e.g. "Singapore" next to Strait of Malacca)
instead of just coordinates.

In [0]:
import numpy as np

ports_pdf = ports.select(
    F.col("PORT_NAME"), F.col("COUNTRY"), F.col("LATITUDE"), F.col("LONGITUDE")
).toPandas()

def nearest_port(lat, lon):
    d2 = (ports_pdf["LATITUDE"] - lat) ** 2 + (ports_pdf["LONGITUDE"] - lon) ** 2
    idx = d2.idxmin()
    return ports_pdf.loc[idx, "PORT_NAME"], ports_pdf.loc[idx, "COUNTRY"]

chokepoints_df["center_lat"] = (chokepoints_df["lat_min"] + chokepoints_df["lat_max"]) / 2
chokepoints_df["center_lon"] = (chokepoints_df["lon_min"] + chokepoints_df["lon_max"]) / 2
nearest = chokepoints_df.apply(lambda r: nearest_port(r["center_lat"], r["center_lon"]), axis=1)
chokepoints_df["nearest_port"] = [n[0] for n in nearest]
chokepoints_df["nearest_port_country"] = [n[1] for n in nearest]

display(chokepoints_df[[
    "chokepoint", "total_traffic", "share_of_ranked_total",
    "nearest_port", "nearest_port_country",
]])

chokepoint,total_traffic,share_of_ranked_total,nearest_port,nearest_port_country
Danish Straits,6408074441700,0.6947721172682166,HUNDESTED,DK
Strait of Malacca,1286868516300,0.13952402891848564,TELUK ANSON,MY
Strait of Hormuz,594608248500,0.06446823230815885,KHAWR KHASAB,OM
Strait of Dover,471353840400,0.05110482230764081,CALAIS,FR
Strait of Gibraltar,155948076200,0.016908101812973853,TANGIER-MEDITERRANEAN,MA
Bab-el-Mandeb,151830609400,0.01646168048119253,ASSAB,ER
Suez Canal,75987978700,0.00823871965418762,EL ISMAILIYA,EG
Bosphorus Strait,71200697400,0.007719676126366622,ISTINYE,TR
Panama Canal,7402795500,8.026211227774331E-4,VACAMONTE,PA


## Save the result

In [0]:
result = spark.createDataFrame(chokepoints_df)
result.write.format("delta").mode("overwrite").saveAsTable("workspace.global_shipping.known_chokepoints_ranked")
print("Saved workspace.global_shipping.known_chokepoints_ranked")

Saved workspace.global_shipping.known_chokepoints_ranked


## What's next

This table has real traffic-density rankings, but density alone still isn't money —
it's back to the same honest caveat from notebook 01, just at a smaller, curated scale
now. Notebook 03 will pull real trade values from UN Comtrade for the countries on
either side of each chokepoint, to turn "how much traffic" into "how many dollars of
trade are actually riding through here" — the number a business reader actually cares
about.